# 🔍 SEC EDGAR Financial Data — Exploration Notebook
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-28

---

### Notebook Objectives
1. Build two master Delta tables (`edgar_sub_all`, `edgar_num_all`)
2. Profile the dataset — row counts, columns, nulls
3. Discover financial tags relevant to ratio analysis
4. Analyze industry and sector distribution
5. Identify filing trends across Pre-COVID, COVID-Impact, Post-COVID periods

### Master Tables
- `edgar_sub_all` → All 28 quarters of filing metadata unified
- `edgar_num_all` → All 28 quarters of financial numbers unified

## Step 1 — Environment Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, isnan, when, lit, sum
import logging

spark = SparkSession.builder.getOrCreate()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("edgar_exploration")

TARGET_YEARS = [2018, 2019, 2020, 2021, 2022, 2023, 2024]

print(f"✅ Spark session ready | Version: {spark.version}")
print(f"✅ Target years: {TARGET_YEARS}")

## Step 2 — Build Master Tables

Unions all 28 quarters into two master Delta tables.  
Uses `unionByName` to handle minor schema differences across quarters.  
Skips missing tables and logs warnings.

In [0]:
def build_master_table(file_type: str, target_years: list) -> None:
    """
    Unions all quarterly Delta tables of a given type into one master table.
    
    Args:
        file_type: 'sub' or 'num'
        target_years: list of years to include
    """
    master_df = None
    skipped = []
    processed = []
    
    for year in target_years:
        for q in range(1, 5):
            table_name = f"edgar_{file_type}_{year}_q{q}"
            try:
                df = spark.table(table_name)
                if master_df is None:
                    master_df = df
                else:
                    master_df = master_df.unionByName(df, allowMissingColumns=True)
                processed.append(table_name)
                logger.info(f"✅ Added: {table_name}")
            except Exception as e:
                skipped.append(table_name)
                logger.warning(f"⚠️ Skipped: {table_name} — {e}")
    
    if master_df is None:
        logger.error("❌ No tables found — master table not created")
        return
    
    # Save as Delta table
    master_table_name = f"edgar_{file_type}_all"
    master_df.write.format("delta").mode("overwrite").saveAsTable(master_table_name)
    
    print(f"\n{'='*50}")
    print(f"✅ Master table created : {master_table_name}")
    print(f"✅ Tables processed     : {len(processed)}")
    print(f"⚠️ Tables skipped       : {len(skipped)}")
    if skipped:
        print(f"   Skipped list        : {skipped}")
    print(f"{'='*50}")


print("✅ build_master_table function defined")

### Step 2a — Build `edgar_sub_all`
> ⏱️ Estimated time: 2-3 minutes

In [0]:
build_master_table("sub", TARGET_YEARS)

### Step 2b — Build `edgar_num_all`
> ⏱️ Estimated time: 20-30 minutes (84M+ rows)

In [0]:
build_master_table("num", TARGET_YEARS)

## Step 3 — Data Profiling

Systematic profiling of both master tables before exploration.
Covers shape, schema, null analysis, and duplicate detection.

### Step 3a — Shape
Row and column counts for both master tables.

In [0]:
def print_shape(table):
    df = spark.table(table)
    print(f"Table: {table} \n Rows: {df.count()} \n Columns: {len(df.columns)}")

print_shape("edgar_sub_all")
print_shape("edgar_num_all")

### Step 3b — Schema Inspection
Column names and data types.  
> ⚠️ Note: EDGAR data is ingested as strings — numeric columns will need casting in Step 3 (Cleaning).

In [0]:
def print_schema_table(table_name):
    df = spark.table(table_name)
    
    print(f"\n{'='*60}")
    print(f"SCHEMA — {table_name}")
    print(f"{'='*60}")
    print(f"{'#':<5} {'Column':<40} {'Data Type':<15}")
    print("-" * 60)
    for i, field in enumerate(df.schema.fields, 1):
        print(f"{i:<5} {field.name:<40} {str(field.dataType):<15}")
    print(f"{'='*60}")
    print(f"Total columns: {len(df.schema.fields)}")

In [0]:
print_schema_table("edgar_sub_all")

In [0]:
print_schema_table("edgar_num_all")

In [0]:
col_descriptions = {
    # SUB table columns
    "adsh": "Accession Number — unique identifier for each SEC filing",
    "cik": "Central Index Key — unique identifier for each company",
    "name": "Company name as registered with SEC",
    "sic": "Standard Industrial Classification code — identifies industry sector",
    "countryba": "Country of business address",
    "stprba": "State or province of business address",
    "cityba": "City of business address",
    "zipba": "Zip code of business address",
    "bas1": "Business address street line 1",
    "filed": "Date the filing was submitted to SEC",
    "period": "COVID period label — Pre-COVID, COVID-Impact, Post-COVID-Recovery",
    "source_year": "Year the filing was ingested from",
    "source_quarter": "Quarter the filing was ingested from",
    "ingestion_timestamp": "Timestamp when record was ingested into Delta table",
    "pipeline_run_id": "Unique ID of the ingestion pipeline run",

    # NUM table columns
    "tag": "XBRL financial tag name — identifies the financial metric",
    "version": "XBRL taxonomy version the tag belongs to",
    "ddate": "Date of the financial data point (end of reporting period)",
    "qtrs": "Number of quarters represented — 0=instant, 1=quarterly, 4=annual",
    "uom": "Unit of measure — USD, shares, pure ratio etc",
    "value": "Numeric value of the financial data point",
    "segments": "Business segment breakdown — NULL means consolidated company level",
}

print(f"{'Column':<45} {'Description'}")
print("=" * 100)
for col_name, desc in col_descriptions.items():
    print(f"{col_name:<45} {desc}")

### Step 3c — Null Analysis
Identifying missing values across key columns in both tables.  
High null counts in critical columns will be flagged for cleaning.

In [0]:
def count_nulls(table_name):
    df = spark.table(table_name)
    null_counts = df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    
    for col_name, null_count in null_counts.items():
        print(f"{col_name}: {null_count}")

In [0]:
count_nulls("edgar_sub_all")

In [0]:
count_nulls("edgar_num_all")

In [0]:
count_duplicates("edgar_sub_all", ["adsh"])